In [ ]:
# %% [markdown]
# ## Scan for TIFF files, collect dimensions & compute summary statistics

# %%
import sys
from pathlib import Path


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'deepbranchai_utils.py').exists() and (candidate / 'README.md').exists():
            return candidate
    raise RuntimeError('Could not find repository root')

REPO_DIR = find_repo_root()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from deepbranchai_utils import get_preprocessing_fix_dir, setup_environment

# Keep this as None to store data and nnU-Net folders under the repository.
# Set it to a large local drive folder to move assets to a data drive.
STORAGE_DIR = None

paths = setup_environment(REPO_DIR, storage_dir=STORAGE_DIR)
DATA_DIR = paths['data']

import pandas as pd
import tifffile

# directory to scan
base_dir = get_preprocessing_fix_dir(DATA_DIR)
records = []

for tif_path in base_dir.rglob('*'):
    if tif_path.suffix.lower() in ('.tif', '.tiff'):
        try:
            with tifffile.TiffFile(str(tif_path)) as tif:
                depth = len(tif.pages)
                first = tif.pages[0].asarray()
                height, width = first.shape[:2]
                channels = first.shape[2] if first.ndim == 3 else 1
        except Exception as e:
            print(f"Could not read {tif_path.name}: {e}")
            continue

        records.append({
            'filename': tif_path.name,
            'width':      width,
            'height':     height,
            'depth':      depth,
            'channels':   channels,
        })

# build a DataFrame
df = pd.DataFrame(records)

# display the full table of filenames and their metrics
print(f"Found {len(df)} TIFF files:\n")
display(df)

# compute and display summary statistics for numeric columns
print("\nSummary statistics for dimensions and channels:")
stats = df[['width', 'height', 'depth', 'channels']].describe().astype(int)
display(stats)

In [ ]:
from pathlib import Path
import numpy as np
import tifffile

base_dir = get_preprocessing_fix_dir(DATA_DIR)
for tif_path in base_dir.rglob('*'):
    if tif_path.suffix.lower() not in ('.tif', '.tiff'):
        continue

    # read all pages into one array of shape (n_pages, H, W)
    with tifffile.TiffFile(str(tif_path)) as tif:
        pages = [page.asarray() for page in tif.pages]
    arr = np.stack(pages, axis=0)

    # sanity check: total pages must be divisible by 3
    n_pages, height, width = arr.shape
    if n_pages % 3 != 0:
        print(f"Skipping {tif_path.name}: page‐count {n_pages} not divisible by 3")
        continue

    # reshape to (Z, 3, H, W) then transpose to (Z, H, W, 3)
    Z = n_pages // 3
    arr = arr.reshape(Z, 3, height, width).transpose(0, 2, 3, 1)

    # write out—in this example we append "_fixed" before the extension
    out_path = tif_path.with_name(tif_path.stem + '_fixed.tif')
    # photometric='rgb' marks each page as an RGB image
    tifffile.imwrite(
        str(out_path),
        arr,
        photometric='rgb',
        metadata=dict(axes='ZYXC')
    )
    print(f"Wrote fixed TIFF to {out_path.name}")

In [ ]:
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt

# Directory containing the fixed TIFFs
base_dir = get_preprocessing_fix_dir(DATA_DIR)

# Find all fixed TIFF files
tif_paths = sorted(base_dir.rglob('*_fixed.tif'))

# Read the first RGB slice from each TIFF into a list of arrays
slices = []
for p in tif_paths:
    arr = tifffile.imread(str(p))       # shape (Z, H, W, C)
    slices.append(arr[0])               # take slice index 0 → shape (H, W, C)

if not slices:
    print("No fixed TIFFs found.")
else:
    n_volumes = len(slices)
    n_channels = slices[0].shape[2]

    # Create a grid of subplots: rows=volumes, cols=channels
    fig, axes = plt.subplots(
        n_volumes,
        n_channels,
        figsize=(n_channels * 4, n_volumes * 4),
        squeeze=False
    )

    for i, (img, p) in enumerate(zip(slices, tif_paths)):
        for c in range(n_channels):
            ax = axes[i, c]
            ax.imshow(img[..., c], cmap='gray')
            ax.axis('off')

            # Label the top row with channel names
            if i == 0:
                ax.set_title(f'Channel {c+1}', fontsize=12)

            # Label the first column with the volume filename
            if c == 0:
                ax.set_ylabel(p.name, fontsize=10)

    plt.tight_layout()
    plt.show()